## Cell 1 — Config: MongoDB connection, target columns, ADLS path

**TODO before running:**
- Confirm `SELECTED_COLUMNS` against the source-of-truth column list (pasted text, not photos — still a placeholder pending confirmation).
- Confirm `MONGO_COLLECTION` name inside the `servicerequest` database.
- Store the Mongo password in a Databricks secret scope — never hardcode it.
- Make `ca.cert` available to the cluster (e.g. via a Unity Catalog Volume or DBFS path) and point `CA_FILE` at it.

**Document shape (from the sample doc):** top-level fields (`serviceRequestNumber`,
`workflowId`, `instruction`, `journey`, `status`, `branch`, ...), a nested `data`
sub-document (`data.customerId`, `data.mode`, ...), and a `customers` array where
each entry has its own nested `approval` and `kycStatus` objects. Cell 3 flattens
all of this and explodes `customers` into one row per customer — see its markdown
note for why, and for how to switch to "primary customer only" if that's wrong.

**Encryption:** intentionally not implemented yet (no org-standard encryption
library identified). `SENSITIVE_COLUMNS` below is just a record of which fields
are flagged `encryption_required = TRUE` in the mapping sheet, for whoever picks
this up later — they currently land in ADLS in plaintext.

In [ ]:
import certifi
from datetime import datetime, timezone
from pymongo import MongoClient
from pyspark.sql import functions as F

# ── MongoDB connection (Xpress Form DB) ──────────────────────────────────────
MONGO_USER        = "xpreaduser"
MONGO_PASSWORD    = dbutils.secrets.get(scope="<SECRET_SCOPE>", key="xpress-mongo-password")
MONGO_HOSTS        = (
    "HBXPRESSFRMDRDB1.hbctxdom.com:28181,"
    "HBXPRBPRDQH01.hbctxdom.com:28181,"
    "HBXPRESSFRMPRDDB1.hbctxdom.com:28181"
)
MONGO_DB           = "servicerequest"
MONGO_COLLECTION   = "service_request"  # TODO confirm exact collection name
REPLICA_SET        = "xpressform"
CA_FILE            = "/Volumes/<catalog>/<schema>/<volume>/ca.cert"  # TODO point at the actual cert location

MONGO_URI = (
    f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_HOSTS}/"
    f"?replicaSet={REPLICA_SET}&tls=true&authSource={MONGO_DB}"
)

# ── Columns required in the final output (from Xpress_Forms_mapping_sheet) ──
# Leaf field names only (no dot paths) — Cell 3 flattens the nested document
# and matches by leaf name. TODO: replace with the verified, pasted column
# list (this is the best-effort OCR read from photos and must be cross-checked
# before this notebook is trusted).
SELECTED_COLUMNS = [
    # paste verified column list here
]

# ── Sensitive columns (encryption_required = TRUE in the mapping sheet) ─────
# Not encrypted yet — see Cell 1 markdown note above. Kept here so the field
# list survives even though no encryption step runs today.
SENSITIVE_COLUMNS = [
    "aadharAddress", "aadharCity", "aadharDOB", "aadharGender", "aadharPincode",
    "accountNumber", "beneficiaryAccountNumber", "beneficiaryIBAN", "beneficiaryPIN",
    "casaAccountNo", "codAadhaarNo", "customerId", "debitAccountNumber", "dob",
    "fdrNumber", "firstName", "fullName", "gender", "ip", "ifscCode", "lastName",
    "lockerAccNumber", "lockerNumber", "middleName", "mobile", "mobileNumber",
    "name", "nameOfBeneficiaryBank", "nameOfReceiver", "panNo", "passportnumber",
    "reBeneficiaryAccountNumber", "remark", "remarks",
]

# ── Primary key / incremental watermark (top-level field) ───────────────────
PRIMARY_KEY_COLUMN  = "serviceRequestNumber"
INCREMENTAL_COLUMN  = "serviceRequestNumber"

# ── ADLS target ───────────────────────────────────────────────────────────────
ADLS_PATH = (
    "abfss://raw@ddiprodvyapaaradlsstd.dfs.core.windows.net/"
    "raw_data/business_banking/merchant/vyapaar/elastic/wow_journey_mongodb_service_request/"
)


## Cell 2 — Determine watermark

Reads the max value of `INCREMENTAL_COLUMN` already landed in ADLS so this run only
pulls new/changed service requests. If no data exists yet at `ADLS_PATH`, this is a
full first load.

In [ ]:
try:
    existing_df  = spark.read.parquet(ADLS_PATH)
    last_value   = existing_df.agg(F.max(INCREMENTAL_COLUMN)).first()[0]
    print(f"Resuming after {INCREMENTAL_COLUMN} = {last_value!r}")
except Exception as e:
    last_value = None
    print(f"No existing data at {ADLS_PATH} — running a full load. ({e})")


## Cell 3 — Fetch from MongoDB in batches, flatten & explode

Documents are nested (`data.*`, and a `customers` array with per-customer
`approval`/`kycStatus` sub-objects — see the sample doc). We can't reliably
project only `SELECTED_COLUMNS` server-side without knowing every field's full
dotted path, so each full document is fetched, then:

1. `flatten()` collapses nested dicts into dot-notated keys.
2. `explode_customers()` produces **one record per entry in `customers`**
   (chosen default — change to `customers[:1]` below for "primary customer
   only" if that's what's actually wanted), merging each customer's fields
   with the parent's.
3. `project()` keeps only `SELECTED_COLUMNS`, matched by their leaf name
   (e.g. `customers.kycStatus.isCustomerChecked` → `isCustomerChecked`).

Still batched (mirrors `windowed_data_loader.ipynb`) to avoid loading the
whole collection into driver memory at once.

In [ ]:
BATCH_SIZE = 20_000


def flatten(d, parent_key=""):
    """Recursively collapse nested dicts into dot-notated keys."""
    items = {}
    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, key))
        else:
            items[key] = v
    return items


def explode_customers(doc):
    """One flat record per entry in `customers` (or one record if absent)."""
    base_fields = {k: v for k, v in doc.items() if k != "customers"}
    flat_base   = flatten(base_fields)
    customers   = doc.get("customers") or [{}]   # change to doc.get("customers", [{}])[:1] for primary-only
    return [{**flat_base, **flatten(cust)} for cust in customers]


def project(record, columns):
    """Keep only the requested columns, matched by their leaf (rightmost) path segment."""
    by_leaf = {k.rsplit(".", 1)[-1]: v for k, v in record.items()}
    return {col: by_leaf.get(col) for col in columns}


client     = MongoClient(MONGO_URI, tlsCAFile=CA_FILE)
collection = client[MONGO_DB][MONGO_COLLECTION]

mongo_filter = {INCREMENTAL_COLUMN: {"$gt": last_value}} if last_value is not None else {}
cursor = collection.find(mongo_filter).batch_size(BATCH_SIZE)

df = None
batch = []

def flush(batch, df):
    if not batch:
        return df
    batch_df = spark.createDataFrame(batch)
    return batch_df if df is None else df.union(batch_df)

for i, doc in enumerate(cursor, start=1):
    for record in explode_customers(doc):
        batch.append(project(record, SELECTED_COLUMNS))
    if len(batch) >= BATCH_SIZE:
        print(f"  flushing batch ending at source document {i:,}")
        df = flush(batch, df)
        batch = []

df = flush(batch, df)
client.close()

if df is None:
    print("No new records since last watermark — nothing to write.")
else:
    print(f"total new rows: {df.count():,}, columns: {len(df.columns)}")


## Cell 4 — Write to ADLS

In [ ]:
if df is not None:
    (
        df.withColumn("ingestion_date", F.current_date())
          .write
          .mode("append")
          .partitionBy("ingestion_date")
          .parquet(ADLS_PATH)
    )
    print(f"wrote {df.count():,} records to {ADLS_PATH}")
